# RF-DETR Mechanics Check — Riot Clips

First run of RF-DETR (Roboflow's real-time DETR object detector) against
the real XD-Violence riot (B4) clips pulled in
`xdviolence_riot_pull.ipynb` (485 clips, validated clean, on Drive at
`My Drive/CVPHG/xdviolence_riot/`).

**This is a mechanics check, not a model result** (see
`docs/findings.md` conventions): COCO-pretrained RF-DETR, no
fine-tuning, run on a small sample of real riot frames to confirm the
detector loads, runs, and produces plausible detections (people,
vehicles) before deciding whether/how to wire it into the riot
classification pipeline (see `docs/roadmap.md` — RF-DETR is meant as a
per-frame *detector* feeding structured features, not a replacement for
the CLIP classifier).

**Requires:** the Drive folder from the pull notebook already populated
(`My Drive/CVPHG/xdviolence_riot/`). Run that notebook first if you
haven't.

## Step 0 — Setup

In [ ]:
!pip install rfdetr supervision opencv-python-headless -q

In [ ]:
import os
import csv
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import supervision as sv
from rfdetr import RFDETRMedium
from rfdetr.assets.coco_classes import COCO_CLASSES
from google.colab import drive

drive.mount('/content/drive')

RIOT_CLIPS_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_riot'
OUT_DIR = '/content/drive/MyDrive/CVPHG/rfdetr_detections'
os.makedirs(OUT_DIR, exist_ok=True)

manifest_path = os.path.join(RIOT_CLIPS_DIR, 'manifest_xdviolence_riot.csv')
manifest = pd.read_csv(manifest_path)
print(f"{len(manifest)} riot clips in manifest")

## Step 1 — Load RF-DETR (COCO-pretrained, no fine-tuning)

First call downloads pretrained weights. Relevant COCO class ids for
this project: `person`=0, `bicycle`=1, `car`=2, `motorcycle`=3, `bus`=5,
`truck`=7 — vehicle ids matter later for the fire/burning-vehicle phase
(`docs/context.md`), not used yet.

In [ ]:
model = RFDETRMedium()

PERSON_CLASS_ID = 0
VEHICLE_CLASS_IDS = {1, 2, 3, 5, 7}  # bicycle, car, motorcycle, bus, truck
DETECTION_THRESHOLD = 0.5

## Step 2 — Mechanics check: run detection on a small sample

Pulls a few evenly-spaced frames from a handful of riot clips and runs
RF-DETR on each frame — validates the model actually runs on real riot
footage before scaling to all 485 clips.

In [ ]:
random.seed(42)
SAMPLE_SIZE = 5
FRAMES_PER_CLIP = 4

sample_rows = manifest.sample(n=min(SAMPLE_SIZE, len(manifest)), random_state=42).to_dict('records')


def sample_frames(video_path, n=FRAMES_PER_CLIP):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    idxs = np.linspace(0, total - 1, min(n, total)).astype(int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame_bgr = cap.read()
        if ok:
            frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

In [ ]:
mechanics_records = []
annotated_examples = []  # (frame, detections, title) for the visual check below

for row in sample_rows:
    video_path = os.path.join(RIOT_CLIPS_DIR, row['id'])
    frames = sample_frames(video_path)
    print(f"{row['id']}: {len(frames)} frames sampled")
    for fi, frame in enumerate(frames):
        detections = model.predict(frame, threshold=DETECTION_THRESHOLD)
        n_person = int((detections.class_id == PERSON_CLASS_ID).sum())
        n_vehicle = int(np.isin(detections.class_id, list(VEHICLE_CLASS_IDS)).sum())
        mean_conf = float(detections.confidence.mean()) if len(detections) else 0.0
        mechanics_records.append(dict(
            video_id=row['id'], frame_idx=fi, num_detections=len(detections),
            num_person=n_person, num_vehicle=n_vehicle, mean_confidence=mean_conf,
        ))
        if len(annotated_examples) < 6:
            annotated_examples.append((frame, detections, f"{row['id'][:30]}...#{fi}"))

mechanics_df = pd.DataFrame(mechanics_records)
mechanics_df

## Step 3 — Sanity checks

RF-DETR ran on every sampled frame with no errors is the main mechanics
check. Beyond that: riot footage should show non-trivial person counts
(these are crowd-scale unrest clips) — a mechanics check returning
all-zero detections across the board would indicate something's wrong
(wrong threshold, preprocessing issue, wrong model), not that riot
footage genuinely has no people in it.

In [ ]:
n = len(mechanics_df)
print(f"{n} frames processed, {mechanics_df['num_detections'].eq(0).sum()} with zero detections")
print(f"Mean person count per frame: {mechanics_df['num_person'].mean():.1f}")
print(f"Mean vehicle count per frame: {mechanics_df['num_vehicle'].mean():.1f}")
print(f"Mean detection confidence: {mechanics_df['mean_confidence'].mean():.3f}")

if mechanics_df['num_detections'].eq(0).all():
    print("\nWARNING: zero detections on every sampled frame -- check threshold/model/frame format before trusting this.")
elif mechanics_df['num_person'].mean() < 1:
    print("\nWARNING: near-zero person detections on riot footage -- worth a manual look at the annotated frames below.")
else:
    print("\nMechanics check passed: detector runs and finds plausible person/vehicle counts on real riot footage.")

## Step 4 — Visual spot-check

Draws RF-DETR's bounding boxes on a handful of sampled frames so you can
eyeball whether the detections make sense (people correctly boxed,
no wildly wrong classes) before this feeds anything downstream.

In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (frame, detections, title) in zip(axes.flat, annotated_examples):
    labels = [f"{COCO_CLASSES[c]} {conf:.2f}" for c, conf in zip(detections.class_id, detections.confidence)]
    annotated = box_annotator.annotate(frame.copy(), detections)
    annotated = label_annotator.annotate(annotated, detections, labels)
    ax.imshow(annotated)
    ax.set_title(title, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
mechanics_csv = os.path.join(OUT_DIR, 'mechanics_check_rfdetr.csv')
mechanics_df.to_csv(mechanics_csv, index=False)
print(f"Saved mechanics-check records to {mechanics_csv}")

## Next steps

- If the sanity checks and visual spot-check look right: scale this up
  to all 485 riot clips (swap `SAMPLE_SIZE` for the full manifest
  length), and decide the actual per-clip feature (e.g. max/mean person
  count across frames, vehicle presence flag) to feed downstream.
- This has not been logged in `docs/findings.md` yet — do that once the
  sanity checks above have actually been eyeballed, not before.
- No fine-tuning attempted here (COCO-pretrained only), consistent with
  every other model in this project so far.